# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Priyanshu-Technologies/flyrank-ML-track/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


### Paper finding 1

The paper reports a finding about which pages are under-capturing clicks relative to their observed search position.

**Methodology question:** How exactly is the label for "under-capturing clicks" constructed? I would want to verify which click-through-rate and position measurements are used, which time window they come from, and whether the label is based only on information available before the prediction or review decision.

This matters because a label that is constructed from the same observation window as the features could make the evaluation look stronger than it would be at a real decision point.

### Paper finding 2

The paper reports model or baseline performance on a held-out evaluation set.

**Methodology question:** Does the validation design match the deployment question? In particular, I would check whether related pages from the same client can appear in both training and evaluation, and whether the evaluation period is strictly separated from the feature period.

A grouped or time-aware split would provide stronger evidence if the intended use is to prioritize pages from clients or periods that were not used during model development.

These are constructive methodology questions rather than judgments about the research. The goal is to understand exactly what the reported numbers support.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In Week 5, the model was evaluated using a client-grouped split, which is already more honest than a random row-level split.

For this audit, I will explicitly compare a random row-level split with the grouped-by-client split.

The random split represents the easier evaluation setting because pages from the same clients can appear in both training and test data.

The grouped split is the primary honest evaluation because all pages from a client are kept entirely within either training or test data.

I will compare Precision@50 under both designs and report the test-set base rate. A gap between the two results is itself evidence about how much the validation design affects measured performance.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


# 1. Load data and create the observed target
df = pd.read_csv(
    "../../data/raw/content_refresh_anonymized.csv"
)

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)



# 2. Feature set from Week 5
feature_columns = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

target = "is_declining_label"



# 3. Precision@50
def precision_at_k(data, score_column, k=50):
    ranked = (
        data
        .sort_values(score_column, ascending=False)
        .head(k)
    )

    return ranked[target].mean()



# 4. Logistic Regression pipeline
def make_model():
    return Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ])


X = df[feature_columns]
y = df[target]



# 5. BEFORE: random row-level split
X_train_random, X_test_random, y_train_random, y_test_random = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
)

random_model = make_model()

random_model.fit(
    X_train_random,
    y_train_random
)

random_scores = random_model.predict_proba(
    X_test_random
)[:, 1]

random_test = pd.DataFrame({
    "score": random_scores,
    target: y_test_random.values
})

random_precision = precision_at_k(
    random_test,
    "score",
    k=50
)



# 6. AFTER: grouped-by-client split
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        df[feature_columns],
        df[target],
        groups=df["client_id"]
    )
)

group_train = df.iloc[train_idx].copy()
group_test = df.iloc[test_idx].copy()

group_model = make_model()

group_model.fit(
    group_train[feature_columns],
    group_train[target]
)

group_scores = group_model.predict_proba(
    group_test[feature_columns]
)[:, 1
]

group_test["model_score"] = group_scores

group_precision = precision_at_k(
    group_test,
    "model_score",
    k=50
)



# 7. Comparison
comparison = pd.DataFrame({
    "validation": [
        "Random row split",
        "Grouped by client"
    ],
    "Precision@50": [
        random_precision,
        group_precision
    ],
    "test_base_rate": [
        y_test_random.mean(),
        group_test[target].mean()
    ]
})

comparison["Precision@50"] = (
    comparison["Precision@50"].round(3)
)

comparison["test_base_rate"] = (
    comparison["test_base_rate"].round(3)
)

display(comparison)



# 8. Validation gap
gap = random_precision - group_precision

print(
    "Random minus grouped Precision@50:",
    round(gap, 3)
)

print(
    "Grouped test clients:",
    group_test["client_id"].nunique()
)

print(
    "Grouped test rows:",
    len(group_test)
)

,validation,Precision@50,test_base_rate
0,Random row split,1.0,0.542
1,Grouped by client,1.0,0.511


Random minus grouped Precision@50: 0.0
Grouped test clients: 7
Grouped test rows: 6163


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I audited the Week-5 feature set for information that would not be legitimately available at the decision point.

The target is `is_declining_label`, derived from `trend_direction`. Therefore `trend_direction` and `trend_pct` must not be model features.

I also exclude identifiers such as `content_id` and `client_id` from the feature set. `client_id` is used only for grouped validation.

I checked the feature list for label-derived fields, future-window fields, and decision-derived fields.

The audit is intended to establish that the model uses observable signals rather than information derived from the outcome being evaluated.

In [2]:
# Leakage audit
# Features actually used by the model
used_features = set(feature_columns)

# Fields that must NEVER be model features
forbidden_features = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "content_id",
    "client_id",
    "report_date"
}

# Check for forbidden fields
leakage_hits = used_features.intersection(
    forbidden_features
)

print("Model features:", len(used_features))
print("Forbidden fields found:", leakage_hits)

assert len(leakage_hits) == 0

print("\n1. Label-derived leakage check: PASSED")
print("   trend_direction and trend_pct are excluded.")


# Check that IDs are not being used as features
assert "content_id" not in used_features
assert "client_id" not in used_features

print("\n2. Identifier leakage check: PASSED")
print("   content_id and client_id are excluded from features.")


# Check that no obvious future-window fields
# are present in the feature list
future_keywords = [
    "future",
    "next_30",
    "next_60",
    "next_90",
    "post_"
]

future_hits = [
    feature
    for feature in feature_columns
    if any(
        keyword in feature.lower()
        for keyword in future_keywords
    )
]

print("\nFuture-window feature candidates:", future_hits)

assert len(future_hits) == 0

print("\n3. Future-window leakage check: PASSED")


# Final feature list
print("\nFinal audited feature list:")

for feature in feature_columns:
    print(" -", feature)

print("\nLEAKAGE AUDIT: PASSED")


Model features: 26
Forbidden fields found: set()

1. Label-derived leakage check: PASSED
   trend_direction and trend_pct are excluded.

2. Identifier leakage check: PASSED
   content_id and client_id are excluded from features.

Future-window feature candidates: []

3. Future-window leakage check: PASSED

Final audited feature list:
 - search_volume
 - competition
 - cpc
 - word_count
 - content_age_days
 - days_since_last_update
 - impressions_90d
 - clicks_90d
 - pageviews_90d
 - sessions_90d
 - users_90d
 - engaged_sessions_90d
 - scroll_events_90d
 - days_with_impressions
 - days_with_sessions
 - impressions_last_30d
 - clicks_last_30d
 - sessions_last_30d
 - impressions_prev_30d
 - clicks_prev_30d
 - sessions_prev_30d
 - ctr
 - avg_position
 - engagement_rate
 - scroll_rate
 - ai_traffic_pct

LEAKAGE AUDIT: PASSED


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Original claim

The Logistic Regression model can identify pages that are likely to decline and can be used to prioritize refreshes.

### Audited claim

The Logistic Regression model produces a ranking score that is associated with the observed declining label in this dataset. On the held-out test clients, I measured its Precision@50 and compared it with the Week-4 rule-based baseline.

The grouped validation audit produced the same Precision@50 as the random split in this run, so this comparison did not show a measurable validation gap.

These results support using the model as directional decision-support for prioritizing pages for human review. They do not establish that the model predicts future performance, that a page will benefit from a refresh, or that any feature causes decline.

A stronger future evaluation would define a genuinely future outcome after the decision point and test whether the ranking helps improve the resulting business or content decision.

In [3]:
# Claim audit
claims = pd.DataFrame({
    "claim": [
        "The model predicts which pages will benefit from a refresh.",
        "The model ranks pages associated with the observed declining label.",
        "The model can provide directional decision-support for human review."
    ],
    "supported_by_current_evidence": [
        "No",
        "Yes",
        "Yes"
    ],
    "reason": [
        "We do not have a future refresh outcome in this evaluation.",
        "Precision@50 was measured against the observed declining label.",
        "The model produces a ranking that can be used to prioritize human review."
    ]
})

display(claims)


# Final safe claim

print("\nFinal public-safe claim:")
print(
    "The Logistic Regression model provides a directional ranking "
    "of pages associated with the observed declining label and can "
    "support prioritization of pages for human review."
)

print(
    "\nThe evaluation does not establish causality or guarantee "
    "that refreshing a ranked page will improve performance."
)

,claim,supported_by_current_evidence,reason
0,The model predicts which pages will benefit fr...,No,We do not have a future refresh outcome in thi...
1,The model ranks pages associated with the obse...,Yes,Precision@50 was measured against the observed...
2,The model can provide directional decision-sup...,Yes,The model produces a ranking that can be used ...



Final public-safe claim:
The Logistic Regression model provides a directional ranking of pages associated with the observed declining label and can support prioritization of pages for human review.

The evaluation does not establish causality or guarantee that refreshing a ranked page will improve performance.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.